In [11]:
import string
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score
)
from wordcloud import WordCloud
import matplotlib.pyplot as plt


In [2]:
# Function to create a dict with each words count 
def count_word(text:str, word_counter:dict):
    clear_text = text.translate(str.maketrans("", "", string.punctuation)).lower()
    clear_text = " ".join(clear_text.split())
    # print(clear_text)
    for word in clear_text.split(" "):
        count = word_counter.get(word)
        if count is None:
            word_counter[word] = 1
        else:
            word_counter[word] = count + 1


# Function to see the first n entries of a dict
def head(dictionary, n):
    return dict(list(dictionary.items())[:n])

## Data Import

In [3]:
# import data 
url = "https://github.com/bozercavdar/kickstarter-project/releases/download/v1.0/kickstarter_data_OSF.csv"
local_path = "dataset.csv"

# UNCOMMENT THE LINE BELOW IF YOU DON'T HAVE THE DATASET IN THE LOCAL DIRECTORY
# df = pd.read_csv(url)
df = pd.read_csv(local_path)

In [4]:
# Filter only those required columns
selected_df = df[['uid', 'blurb', 'goal', 'state', 'usd_pledged', 'category']]
# Add column that states if a project is succesful or not
selected_df['success'] = selected_df['state'] == 'successful'
selected_df.head()

/tmp/ipykernel_59451/2469478530.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df['success'] = selected_df['state'] == 'successful'


,uid,blurb,goal,state,usd_pledged,category,success
0,1,This project is designed to help protect the e...,2500.0,failed,0.0,Music,False
1,2,Help us built a sustainable studio & eliminate...,25000.0,failed,1.0,Technology,False
2,3,"""If I paint something, I don't want to have to...",5000.0,failed,5.0,Art,False
3,4,Our free app will allow you pool reservations ...,12000.0,failed,0.0,Food,False
4,5,Prohibition themed Gastro Pub and After Dark S...,20000.0,failed,0.0,Food,False


## Create Lexicons

In [5]:
# The idea is to find those words that play a distinctive role in success or failure
# To prevent data leakage, split the data into 2. 
# Create the lexicons looking at the first part. Do the analysis on the second part.

In [6]:
# Split the dataset into 2
X = selected_df.copy()
y = pd.DataFrame(np.zeros(shape=(len(X))))

df_first, df_second, _ , _ = train_test_split(
    X, y, test_size=0.50, random_state=41
)

In [7]:
# Find the count of words without stopwords on only succesful projects 
succesful_word_dict = {}
for i, row in df_first.iterrows():
    # print(row)
    if row['success']:
        count_word(row['blurb'], succesful_word_dict)

successful_filtered_word_dict = {w: c for w, c in succesful_word_dict.items() if w not in ENGLISH_STOP_WORDS}
successful_filtered_word_dict = dict(sorted(successful_filtered_word_dict.items(), key=lambda item: item[1], reverse=True))
head(successful_filtered_word_dict, 30)

{'new': 5088,
 'help': 4955,
 'book': 3334,
 'album': 3054,
 'music': 2523,
 'art': 2463,
 'world': 2268,
 'film': 2247,
 'make': 1917,
 'life': 1806,
 'need': 1664,
 'love': 1601,
 'series': 1596,
 'story': 1595,
 'project': 1525,
 'short': 1385,
 'time': 1268,
 'game': 1246,
 'inspired': 1239,
 'record': 1213,
 'collection': 1200,
 'featuring': 1165,
 'create': 1146,
 'original': 1138,
 'set': 1086,
 'people': 1068,
 'enamel': 1065,
 'support': 1057,
 'songs': 1022,
 'want': 986}

In [8]:
# Find the count of words without stopwords on only failed projects 
failed_word_dict = {}
for i, row in df_first.iterrows():
    # print(row)
    if not row['success']:
        count_word(row['blurb'], failed_word_dict)

failed_filtered_word_dict = {w: c for w, c in failed_word_dict.items() if w not in ENGLISH_STOP_WORDS}
failed_filtered_word_dict = dict(sorted(failed_filtered_word_dict.items(), key=lambda item: item[1], reverse=True))
head(failed_filtered_word_dict, 30)

{'help': 3309,
 'new': 2584,
 'music': 1997,
 'world': 1735,
 'want': 1723,
 'make': 1698,
 'art': 1655,
 'create': 1607,
 'people': 1507,
 'project': 1401,
 'need': 1393,
 'life': 1349,
 'book': 1272,
 'film': 1260,
 'love': 1166,
 'food': 1140,
 'creating': 1075,
 'album': 1057,
 'like': 1014,
 'game': 974,
 'time': 970,
 'app': 858,
 'based': 853,
 'bring': 849,
 'community': 842,
 'im': 825,
 'way': 809,
 'unique': 787,
 'video': 761,
 'series': 742}

In [9]:
# Looking at the count of the words in each scenario create lexicons
positively_distinctive_words = [
    "book",
    "album",
    "series",
    "story",
    "short",
    "inspired",
    "record",
    "collection",
    "featuring",
    "original",
    "set",
    "enamel",
    "support",
    "songs",
    "creative",      # synonym expansion
    "illustrated",   # common in successful creative projects
    "chapter",       # related to book/story
    "soundtrack",    # related to music/album/record
    "limited-run",   # often associated with successful art/collectibles
    "crafted"        # common in successful physical-art Kickstarters
]

negatively_distinctive_words = [
    "food",
    "creating",
    "like",
    "app",
    "based",
    "bring",
    "community",
    "im",
    "way",
    "unique",
    "video",
    "want",          # significantly higher in failed campaigns
    "people",        # overrepresented in failed campaigns
    "create",        # also higher in failed
    "concept",       # often used in vague/project-not-ready pitches
    "idea",          # commonly appears in weak or unvalidated projects
    "plan",          # generic / non-specific wording
    "prototype-needed", # indicates unprepared campaigns
    "trying",        # weak commitment language
    "hope"           # similar "wishful" tone common in failures
]

## Analysis

In [14]:
# Function to compute the number of dictionary matches per 100 words.
def compute_score(text, dictionary):
    words = re.findall(r"\b\w+\b", text)
    word_count = len(words) if len(words) > 0 else 1

    count = 0
    for term in dictionary:
        # If phrase, check as substring
        if " " in term:
            if term in text:
                count += 1
        else:
            # Single-word matching
            count += words.count(term)

    return (count / word_count) * 100  # density per 100 words

In [15]:
# Add the percentages as new columns
df_second["positive_score"] = df_second["blurb"].apply(lambda x: compute_score(x, positively_distinctive_words))
df_second["negative_score"] = df_second["blurb"].apply(lambda x: compute_score(x, negatively_distinctive_words))
df_second

,uid,blurb,goal,state,usd_pledged,category,success,positive_score,negative_score
155006,189791,Spend less time searching your purse and more ...,20000.0,successful,23088.608880,Fashion,True,0.000000,0.000000
69020,77192,Kevin Jenkins will choreograph and direct a ne...,3000.0,successful,3635.318822,web,True,5.882353,5.882353
75028,83929,Finalist in the Telio National Design Competit...,800.0,successful,1126.748389,Fashion,True,0.000000,0.000000
19323,21429,What are the real stories behind these unique ...,5000.0,failed,450.000000,Journalism,False,0.000000,4.545455
22301,24722,"High cost-performance, Fastest, Highest accura...",50000.0,failed,37382.000000,Technology,False,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
114618,131952,Introducing the drum caps that connect to gard...,3000.0,successful,4205.000000,Technology,True,0.000000,0.000000
131512,154738,A follow up double album to the hugely popular...,25000.0,successful,60163.781580,Music,True,9.090909,0.000000
85203,95710,"Integrated GoPro mounting system, water resist...",12000.0,successful,12656.000000,Technology,True,0.000000,0.000000
23842,26460,"Each episode presents: ""A Day in the Life of a...",61060.0,failed,130.000000,Film & Video,False,0.000000,0.000000


In [16]:
# Create train and test sets to be used in a classification algorithm
X = df_second[["positive_score", "negative_score"]].copy()
y = df_second["success"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=41, stratify=y
)

In [17]:
# peek to train set
pd.concat([X_train.head(), y_train.head()], axis=1)

,positive_score,negative_score,success
54152,4.761905,4.761905,False
159607,0.000000,0.000000,True
107282,4.761905,0.000000,True
150992,0.000000,0.000000,True
22150,4.545455,0.000000,False


In [18]:
# Fit and predict a classification algorithm
logit = LogisticRegression(max_iter=500)
logit.fit(X_train, y_train)

y_pred = logit.predict(X_test)
y_pred_prob = logit.predict_proba(X_test)[:, 1]

In [19]:
# Percentage of predictions as successful  
y_pred.mean()

np.float64(0.8068077660194983)

In [20]:
# Examine the results
print("\n=== Logistic Regression Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Coefficients
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coef": logit.coef_[0]
})
print("\n=== Coefficients ===")
print(coef_df)


=== Logistic Regression Performance ===
Accuracy: 0.6166986084492959
Precision: 0.6249935450555125
Recall: 0.8619756427604871
AUC: 0.6112887051038113

Confusion Matrix:
 [[ 2699  7262]
 [ 1938 12103]]

=== Coefficients ===
          feature      coef
0  positive_score  0.107660
1  negative_score -0.089529
